## TPCH SF10 — Silver & Gold Layer Transformations

**Source**: `nickakin_uc.tpch_sf10` (bronze)  
**Target**: `nickakin_uc.benchmark_tests` (silver `_silver` suffix, gold `_gold` suffix)

### Silver Tables (Cleaned & Enriched)
| Silver Table | Source Tables | Key Transformations |
| --- | --- | --- |
| `customer_silver` | customer, nation, region | Enriched with nation/region names, lowercased column names, cast key types |
| `supplier_silver` | supplier, nation, region | Enriched with nation/region names, lowercased column names |
| `part_silver` | part | Standardized types, lowercased columns, trimmed strings |
| `partsupp_silver` | partsupp, part, supplier | Enriched with part name/brand, supplier name, computed inventory value |
| `orders_silver` | orders, customer | Enriched with customer name/segment, extracted order year/month |
| `lineitem_silver` | lineitem, orders, part, supplier | Computed net revenue, shipping delay days, enriched with part/supplier/order details |

### Gold Tables (Aggregated & Business-Ready)
| Gold Table | Silver Sources | Description |
| --- | --- | --- |
| `revenue_by_region_gold` | lineitem_silver, orders_silver, customer_silver | Revenue, order count, and avg discount by region and nation |
| `customer_lifetime_value_gold` | orders_silver, lineitem_silver | Per-customer total orders, revenue, AOV, first/last order dates |
| `supplier_performance_gold` | lineitem_silver, supplier_silver | Per-supplier revenue, qty shipped, avg shipping delay, on-time rate |
| `monthly_sales_gold` | orders_silver, lineitem_silver | Time-series monthly revenue, order count, and avg order value |
| `part_sales_gold` | lineitem_silver | Per-part total quantity, revenue, order count, avg discount, return rate |

In [0]:
# ---------------------------------------------------------------------------- #
#  Snowpark Connect for Spark — Session Initialization (Install Snowpark Connect & enable the 2 lines)
# ---------------------------------------------------------------------------- #

# pip install snowpark-connect[jdk]

# import snowflake.snowpark_connect
# spark = snowflake.snowpark_connect.init_spark_session()



# ---------------------------------------------------------------------------- #
#  Configuration
# ---------------------------------------------------------------------------- #

SOURCE_CATALOG = "nickakin_uc"
SOURCE_SCHEMA  = "tpch_sf100"
TARGET_CATALOG = "nickakin_uc"
TARGET_SCHEMA  = "benchmark_tests"

def src(table: str) -> str:
    """Return fully-qualified source table name."""
    return f"{SOURCE_CATALOG}.{SOURCE_SCHEMA}.{table}"

def tgt(table: str) -> str:
    """Return fully-qualified target table name with _silver suffix."""
    return f"{TARGET_CATALOG}.{TARGET_SCHEMA}.{table}_silver"

# Ensure target schema exists
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {TARGET_CATALOG}.{TARGET_SCHEMA}")

print(f"Source : {SOURCE_CATALOG}.{SOURCE_SCHEMA}")
print(f"Target : {TARGET_CATALOG}.{TARGET_SCHEMA}")

In [0]:
import time, datetime

_pipeline_start = time.time()
print(f"⏱️  Pipeline started at {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

In [0]:
# ---------------------------------------------------------------------------- #
#  customer_silver
#  Enrich customers with nation and region names
# ---------------------------------------------------------------------------- #
from pyspark.sql import functions as F

customer = spark.table(src("customer"))
nation   = spark.table(src("nation"))
region   = spark.table(src("region"))

customer_silver = (
    customer
    .join(nation, customer["C_NATIONKEY"] == nation["N_NATIONKEY"], "left")
    .join(region, nation["N_REGIONKEY"] == region["R_REGIONKEY"], "left")
    .select(
        F.col("C_CUSTKEY").cast("long").alias("customer_key"),
        F.trim(F.col("C_NAME")).alias("customer_name"),
        F.trim(F.col("C_ADDRESS")).alias("address"),
        F.trim(F.col("C_PHONE")).alias("phone"),
        F.col("C_ACCTBAL").cast("double").alias("account_balance"),
        F.trim(F.col("C_MKTSEGMENT")).alias("market_segment"),
        F.trim(F.col("N_NAME")).alias("nation_name"),
        F.trim(F.col("R_NAME")).alias("region_name"),
        F.current_timestamp().alias("_etl_loaded_at"),
    )
)

(
    customer_silver
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tgt("customer"))
)

print(f"\u2705 {tgt('customer')} written — {spark.table(tgt('customer')).count():,} rows")

In [0]:
# ---------------------------------------------------------------------------- #
#  supplier_silver
#  Enrich suppliers with nation and region names
# ---------------------------------------------------------------------------- #

supplier = spark.table(src("supplier"))

supplier_silver = (
    supplier
    .join(nation, supplier["S_NATIONKEY"] == nation["N_NATIONKEY"], "left")
    .join(region, nation["N_REGIONKEY"] == region["R_REGIONKEY"], "left")
    .select(
        F.col("S_SUPPKEY").cast("long").alias("supplier_key"),
        F.trim(F.col("S_NAME")).alias("supplier_name"),
        F.trim(F.col("S_ADDRESS")).alias("address"),
        F.trim(F.col("S_PHONE")).alias("phone"),
        F.col("S_ACCTBAL").cast("double").alias("account_balance"),
        F.trim(F.col("N_NAME")).alias("nation_name"),
        F.trim(F.col("R_NAME")).alias("region_name"),
        F.current_timestamp().alias("_etl_loaded_at"),
    )
)

(
    supplier_silver
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tgt("supplier"))
)

print(f"\u2705 {tgt('supplier')} written — {spark.table(tgt('supplier')).count():,} rows")

In [0]:
# ---------------------------------------------------------------------------- #
#  part_silver
#  Standardize types, trim strings, lowercase column names
# ---------------------------------------------------------------------------- #

part = spark.table(src("part"))

part_silver = (
    part
    .select(
        F.col("P_PARTKEY").cast("long").alias("part_key"),
        F.trim(F.col("P_NAME")).alias("part_name"),
        F.trim(F.col("P_MFGR")).alias("manufacturer"),
        F.trim(F.col("P_BRAND")).alias("brand"),
        F.trim(F.col("P_TYPE")).alias("part_type"),
        F.col("P_SIZE").cast("int").alias("size"),
        F.trim(F.col("P_CONTAINER")).alias("container"),
        F.col("P_RETAILPRICE").cast("double").alias("retail_price"),
        F.current_timestamp().alias("_etl_loaded_at"),
    )
)

(
    part_silver
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tgt("part"))
)

print(f"\u2705 {tgt('part')} written — {spark.table(tgt('part')).count():,} rows")

In [0]:
# ---------------------------------------------------------------------------- #
#  partsupp_silver
#  Enrich with part name/brand and supplier name; compute inventory value
# ---------------------------------------------------------------------------- #

partsupp = spark.table(src("partsupp"))

partsupp_silver = (
    partsupp
    .join(part, partsupp["PS_PARTKEY"] == part["P_PARTKEY"], "left")
    .join(supplier, partsupp["PS_SUPPKEY"] == supplier["S_SUPPKEY"], "left")
    .select(
        F.col("PS_PARTKEY").cast("long").alias("part_key"),
        F.col("PS_SUPPKEY").cast("long").alias("supplier_key"),
        F.trim(F.col("P_NAME")).alias("part_name"),
        F.trim(F.col("P_BRAND")).alias("brand"),
        F.trim(F.col("S_NAME")).alias("supplier_name"),
        F.col("PS_AVAILQTY").cast("int").alias("available_qty"),
        F.col("PS_SUPPLYCOST").cast("double").alias("supply_cost"),
        (F.col("PS_AVAILQTY") * F.col("PS_SUPPLYCOST")).cast("double").alias("inventory_value"),
        F.current_timestamp().alias("_etl_loaded_at"),
    )
)

(
    partsupp_silver
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tgt("partsupp"))
)

print(f"\u2705 {tgt('partsupp')} written — {spark.table(tgt('partsupp')).count():,} rows")

In [0]:
# ---------------------------------------------------------------------------- #
#  orders_silver
#  Enrich with customer info; extract date parts for analytics
# ---------------------------------------------------------------------------- #

orders = spark.table(src("orders"))

orders_silver = (
    orders
    .join(customer, orders["O_CUSTKEY"] == customer["C_CUSTKEY"], "left")
    .select(
        F.col("O_ORDERKEY").cast("long").alias("order_key"),
        F.col("O_CUSTKEY").cast("long").alias("customer_key"),
        F.trim(F.col("C_NAME")).alias("customer_name"),
        F.trim(F.col("C_MKTSEGMENT")).alias("market_segment"),
        F.trim(F.col("O_ORDERSTATUS")).alias("order_status"),
        F.col("O_TOTALPRICE").cast("double").alias("total_price"),
        F.col("O_ORDERDATE").alias("order_date"),
        F.year("O_ORDERDATE").alias("order_year"),
        F.month("O_ORDERDATE").alias("order_month"),
        F.quarter("O_ORDERDATE").alias("order_quarter"),
        F.trim(F.col("O_ORDERPRIORITY")).alias("order_priority"),
        F.trim(F.col("O_CLERK")).alias("clerk"),
        F.col("O_SHIPPRIORITY").cast("int").alias("ship_priority"),
        F.current_timestamp().alias("_etl_loaded_at"),
    )
)

(
    orders_silver
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tgt("orders"))
)

print(f"\u2705 {tgt('orders')} written — {spark.table(tgt('orders')).count():,} rows")

In [0]:
# ---------------------------------------------------------------------------- #
#  lineitem_silver
#  Core fact table with computed revenue, shipping delay, and enrichments
# ---------------------------------------------------------------------------- #

lineitem = spark.table(src("lineitem"))

lineitem_silver = (
    lineitem
    .join(orders, lineitem["L_ORDERKEY"] == orders["O_ORDERKEY"], "left")
    .join(part, lineitem["L_PARTKEY"] == part["P_PARTKEY"], "left")
    .join(supplier, lineitem["L_SUPPKEY"] == supplier["S_SUPPKEY"], "left")
    .select(
        # Keys
        F.col("L_ORDERKEY").cast("long").alias("order_key"),
        F.col("L_LINENUMBER").cast("int").alias("line_number"),
        F.col("L_PARTKEY").cast("long").alias("part_key"),
        F.col("L_SUPPKEY").cast("long").alias("supplier_key"),

        # Enrichment from orders
        F.col("O_ORDERDATE").alias("order_date"),
        F.trim(F.col("O_ORDERSTATUS")).alias("order_status"),

        # Enrichment from part
        F.trim(F.col("P_NAME")).alias("part_name"),
        F.trim(F.col("P_BRAND")).alias("brand"),
        F.trim(F.col("P_TYPE")).alias("part_type"),

        # Enrichment from supplier
        F.trim(F.col("S_NAME")).alias("supplier_name"),

        # Measures
        F.col("L_QUANTITY").cast("double").alias("quantity"),
        F.col("L_EXTENDEDPRICE").cast("double").alias("extended_price"),
        F.col("L_DISCOUNT").cast("double").alias("discount"),
        F.col("L_TAX").cast("double").alias("tax"),

        # Computed: net_revenue = extended_price * (1 - discount) * (1 + tax)
        (
            F.col("L_EXTENDEDPRICE")
            * (F.lit(1) - F.col("L_DISCOUNT"))
            * (F.lit(1) + F.col("L_TAX"))
        ).cast("double").alias("net_revenue"),

        # Shipping fields
        F.trim(F.col("L_RETURNFLAG")).alias("return_flag"),
        F.trim(F.col("L_LINESTATUS")).alias("line_status"),
        F.col("L_SHIPDATE").alias("ship_date"),
        F.col("L_COMMITDATE").alias("commit_date"),
        F.col("L_RECEIPTDATE").alias("receipt_date"),
        F.trim(F.col("L_SHIPMODE")).alias("ship_mode"),
        F.trim(F.col("L_SHIPINSTRUCT")).alias("ship_instruct"),

        # Computed: days between commit and actual ship
        F.datediff(F.col("L_SHIPDATE"), F.col("L_COMMITDATE")).alias("shipping_delay_days"),

        # ETL metadata
        F.current_timestamp().alias("_etl_loaded_at"),
    )
)

# Drop existing streaming table if present, then write
spark.sql(f"DROP TABLE IF EXISTS {tgt('lineitem')}")

(
    lineitem_silver
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tgt("lineitem"))
)

print(f"\u2705 {tgt('lineitem')} written — {spark.table(tgt('lineitem')).count():,} rows")

In [0]:
# ---------------------------------------------------------------------------- #
#  Verification — row counts for all silver tables
# ---------------------------------------------------------------------------- #

silver_tables = ["customer", "supplier", "part", "partsupp", "orders", "lineitem"]

rows = []
for t in silver_tables:
    fqn = tgt(t)
    cnt = spark.table(fqn).count()
    rows.append((fqn, cnt))

verification_df = spark.createDataFrame(rows, ["table_name", "row_count"])
display(verification_df)

---
## Gold Layer — Aggregated Business Tables

All gold tables read from the `_silver` tables above and write to `nickakin_uc.benchmark_tests` with a `_gold` suffix.

| Gold Table | Silver Sources | Description |
| --- | --- | --- |
| `revenue_by_region_gold` | lineitem_silver, customer_silver | Revenue, order count, and avg discount by region and nation |
| `customer_lifetime_value_gold` | orders_silver, lineitem_silver | Per-customer total orders, revenue, AOV, first/last order dates |
| `supplier_performance_gold` | lineitem_silver, supplier_silver | Per-supplier revenue, qty shipped, avg shipping delay, on-time rate |
| `monthly_sales_gold` | orders_silver, lineitem_silver | Time-series monthly revenue, order count, and avg order value |
| `part_sales_gold` | lineitem_silver | Per-part total quantity, revenue, order count, avg discount |

In [0]:
# ---------------------------------------------------------------------------- #
#  Gold layer helper
# ---------------------------------------------------------------------------- #

def gold(table: str) -> str:
    """Return fully-qualified target table name with _gold suffix."""
    return f"{TARGET_CATALOG}.{TARGET_SCHEMA}.{table}_gold"

# Load silver tables into DataFrames
customer_s  = spark.table(tgt("customer"))
supplier_s  = spark.table(tgt("supplier"))
orders_s    = spark.table(tgt("orders"))
lineitem_s  = spark.table(tgt("lineitem"))
part_s      = spark.table(tgt("part"))
partsupp_s  = spark.table(tgt("partsupp"))

print("Silver tables loaded for gold layer transformations.")

In [0]:
# ---------------------------------------------------------------------------- #
#  revenue_by_region_gold
#  Revenue, order count, avg discount by region and nation
# ---------------------------------------------------------------------------- #

revenue_by_region_gold = (
    lineitem_s.alias("li")
    .join(
        orders_s.select("order_key", "customer_key").alias("o"),
        F.col("li.order_key") == F.col("o.order_key"),
        "inner"
    )
    .join(
        customer_s.select("customer_key", "nation_name", "region_name").alias("c"),
        F.col("o.customer_key") == F.col("c.customer_key"),
        "inner"
    )
    .groupBy("region_name", "nation_name")
    .agg(
        F.round(F.sum("net_revenue"), 2).alias("total_revenue"),
        F.countDistinct(F.col("li.order_key")).alias("total_orders"),
        F.round(F.sum("quantity"), 2).alias("total_quantity"),
        F.round(F.avg("discount"), 4).alias("avg_discount"),
        F.round(F.avg("net_revenue"), 2).alias("avg_line_revenue"),
    )
    .withColumn("_etl_loaded_at", F.current_timestamp())
)

(
    revenue_by_region_gold
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold("revenue_by_region"))
)

print(f"\u2705 {gold('revenue_by_region')} — {spark.table(gold('revenue_by_region')).count():,} rows")
display(spark.table(gold("revenue_by_region")).orderBy(F.desc("total_revenue")))

In [0]:
# ---------------------------------------------------------------------------- #
#  customer_lifetime_value_gold
#  Per-customer: total orders, total revenue, AOV, first/last order
# ---------------------------------------------------------------------------- #

# Aggregate line-item revenue per order
order_revenue = (
    lineitem_s
    .groupBy("order_key")
    .agg(
        F.sum("net_revenue").alias("order_revenue"),
        F.sum("quantity").alias("order_qty"),
    )
)

customer_ltv_gold = (
    orders_s
    .join(order_revenue, on="order_key", how="left")
    .groupBy(
        "customer_key", "customer_name", "market_segment"
    )
    .agg(
        F.count("order_key").alias("total_orders"),
        F.round(F.sum("order_revenue"), 2).alias("lifetime_revenue"),
        F.round(F.avg("order_revenue"), 2).alias("avg_order_value"),
        F.round(F.sum("order_qty"), 2).alias("lifetime_quantity"),
        F.min("order_date").alias("first_order_date"),
        F.max("order_date").alias("last_order_date"),
        F.datediff(F.max("order_date"), F.min("order_date")).alias("customer_tenure_days"),
    )
    .withColumn("_etl_loaded_at", F.current_timestamp())
)

(
    customer_ltv_gold
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold("customer_lifetime_value"))
)

print(f"\u2705 {gold('customer_lifetime_value')} — {spark.table(gold('customer_lifetime_value')).count():,} rows")
display(spark.table(gold("customer_lifetime_value")).orderBy(F.desc("lifetime_revenue")).limit(10))

In [0]:
# ---------------------------------------------------------------------------- #
#  supplier_performance_gold
#  Per-supplier: revenue, qty shipped, avg shipping delay, on-time %
# ---------------------------------------------------------------------------- #

supplier_perf_gold = (
    lineitem_s
    .join(
        supplier_s.select("supplier_key", "nation_name", "region_name"),
        on="supplier_key",
        how="inner"
    )
    .groupBy(
        "supplier_key", "supplier_name", "nation_name", "region_name"
    )
    .agg(
        F.round(F.sum("net_revenue"), 2).alias("total_revenue"),
        F.round(F.sum("quantity"), 2).alias("total_quantity"),
        F.countDistinct("order_key").alias("total_orders"),
        F.round(F.avg("shipping_delay_days"), 2).alias("avg_shipping_delay_days"),
        F.round(
            F.sum(F.when(F.col("shipping_delay_days") <= 0, 1).otherwise(0))
            / F.count("*") * 100, 2
        ).alias("on_time_pct"),
        F.round(F.avg("discount"), 4).alias("avg_discount_given"),
    )
    .withColumn("_etl_loaded_at", F.current_timestamp())
)

(
    supplier_perf_gold
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold("supplier_performance"))
)

print(f"\u2705 {gold('supplier_performance')} — {spark.table(gold('supplier_performance')).count():,} rows")
display(spark.table(gold("supplier_performance")).orderBy(F.desc("total_revenue")).limit(10))

In [0]:
# ---------------------------------------------------------------------------- #
#  monthly_sales_gold
#  Time-series: revenue, orders, avg order value by year/month
# ---------------------------------------------------------------------------- #

# Aggregate line-item revenue per order (reuse if available)
order_rev = (
    lineitem_s
    .groupBy("order_key")
    .agg(F.sum("net_revenue").alias("order_net_revenue"))
)

monthly_sales_gold = (
    orders_s
    .join(order_rev, on="order_key", how="left")
    .groupBy("order_year", "order_month", "order_quarter")
    .agg(
        F.count("order_key").alias("total_orders"),
        F.round(F.sum("order_net_revenue"), 2).alias("total_revenue"),
        F.round(F.avg("order_net_revenue"), 2).alias("avg_order_value"),
        F.countDistinct("customer_key").alias("unique_customers"),
        F.round(F.sum("total_price"), 2).alias("total_gross_price"),
    )
    .withColumn("_etl_loaded_at", F.current_timestamp())
    .orderBy("order_year", "order_month")
)

(
    monthly_sales_gold
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold("monthly_sales"))
)

print(f"\u2705 {gold('monthly_sales')} — {spark.table(gold('monthly_sales')).count():,} rows")
display(spark.table(gold("monthly_sales")).orderBy("order_year", "order_month"))

In [0]:
# ---------------------------------------------------------------------------- #
#  part_sales_gold
#  Per-part: total qty sold, revenue, order count, avg discount
# ---------------------------------------------------------------------------- #

part_sales_gold = (
    lineitem_s
    .groupBy(
        "part_key", "part_name", "brand", "part_type"
    )
    .agg(
        F.round(F.sum("quantity"), 2).alias("total_quantity_sold"),
        F.round(F.sum("net_revenue"), 2).alias("total_revenue"),
        F.countDistinct("order_key").alias("total_orders"),
        F.round(F.avg("extended_price"), 2).alias("avg_unit_price"),
        F.round(F.avg("discount"), 4).alias("avg_discount"),
        F.round(
            F.sum(F.when(F.col("return_flag") == "R", 1).otherwise(0))
            / F.count("*") * 100, 2
        ).alias("return_rate_pct"),
    )
    .withColumn("_etl_loaded_at", F.current_timestamp())
)

(
    part_sales_gold
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold("part_sales"))
)

print(f"\u2705 {gold('part_sales')} — {spark.table(gold('part_sales')).count():,} rows")
display(spark.table(gold("part_sales")).orderBy(F.desc("total_revenue")).limit(10))

In [0]:
# ---------------------------------------------------------------------------- #
#  Final Verification — all gold table row counts
# ---------------------------------------------------------------------------- #

gold_tables = [
    "revenue_by_region", "customer_lifetime_value",
    "supplier_performance", "monthly_sales", "part_sales"
]

gold_rows = []
for t in gold_tables:
    fqn = gold(t)
    cnt = spark.table(fqn).count()
    gold_rows.append((fqn, cnt))

gold_verification_df = spark.createDataFrame(gold_rows, ["table_name", "row_count"])
display(gold_verification_df)

In [0]:
_pipeline_end = time.time()
_elapsed = _pipeline_end - _pipeline_start

print(f"⏱️  Pipeline finished at {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"✅  Total execution time: {_elapsed:,.2f} seconds ({_elapsed/60:,.1f} minutes)")